In [ ]:
import os, sys, warnings, requests, pandas as pd
from bs4 import BeautifulSoup
warnings.filterwarnings("ignore")
sys.path.append('/dfs/user/hanchenw/BioAgentOS')

from bioagentos.agent.codeact import codeact_agent
from bioagentos.utils import load_pkl

In [ ]:
tool_index = "all_250324"
llm_name =  "claude-3-7-sonnet-20250219"
tools = load_pkl(f'/dfs/user/hanchenw/BioAgentOS/bioagentos/tool/{tool_index}.pkl')

In [ ]:
coding_tool = {}
coding_tool['bioagentos.tool.support_tools'] = tools['bioagentos.tool.support_tools']

In [ ]:
# codeact
agent = codeact_agent(coding_tool, llm_name)
agent.configure()

In [ ]:
def get_geo_metadata(gse_id: str):
    """
    Fetch metadata for a GEO dataset given its GSE ID.

    Args:
        gse_id (str): GEO Series accession number (e.g. 'GSE122080')

    Returns:
        dict: Dictionary containing:
            - title: Study title
            - summary: Study description/summary
            - overall_design: Study design information

    Raises:
        ValueError: If GSE ID is invalid or data cannot be retrieved
        ConnectionError: If there are network issues
    """

    # Validate GSE ID format
    if not gse_id.startswith("GSE") or not gse_id[3:].isdigit():
        raise ValueError(
            "Invalid GSE ID format. Must start with 'GSE' followed by numbers"
        )

    # Use GEO's direct API
    url = f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gse_id}&targ=self&form=xml&view=quick"

    try:
        # Fetch the record
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "xml")

        # Find the Series element
        series = soup.find("Series")
        if not series:
            raise ValueError(f"No metadata found for GSE ID: {gse_id}")

        # Extract metadata with error handling
        try:
            title = series.find("Title").text.strip()
        except (AttributeError, TypeError):
            title = ""

        try:
            summary = series.find("Summary").text.strip()
        except (AttributeError, TypeError):
            summary = ""

        try:
            design = series.find("Overall-Design").text.strip()
        except (AttributeError, TypeError):
            design = ""

        # Validate that we got at least some data
        if not any([title, summary, design]):
            raise ValueError(f"No metadata content found for GSE ID: {gse_id}")

        return {
            "title": title,
            "summary": summary,
            "overall_design": design,
        }

    except requests.exceptions.RequestException as e:
        raise ConnectionError(f"Failed to fetch data from GEO: {str(e)}")
    except Exception as e:
        raise ValueError(f"Failed to parse GEO response: {str(e)}")


# Example usage:
metadata = get_geo_metadata("GSE99999")
print("Title:", metadata["title"])
print("\nSummary:", metadata["summary"])
print("\nStudy Design:", metadata["overall_design"])

In [ ]:
pd.read_csv("/dfs/user/hanchenw/Agent_PG/clockbase_agent/data/clock_output/human_rna_age_short.csv")

In [ ]:
ls ../data

In [ ]:
study_id = "GSE100007" # "GSE99999"
study_design = get_geo_metadata(study_id)

base_dir = "/dfs/user/hanchenw/Agent_PG/clockbase_agent/"
data_dir = f"{base_dir}/data"
results_dir = f"{base_dir}/result_hs_rna"


task_intention = f'''
Analyze the aging effects of conditions from study {study_id}.

Workflow:

1. Data Acquisition and Preparation
    * Load dataset overview from: `{data_dir}/dataset_level/gse_rna_hs.csv` (Filter by "accession" column to locate {study_id})
    * Extract metadata from: `{data_dir}/GEO_hs_rna/processed_metadata_preharmonize/{study_id}.csv`
    * Import pre-calculated aging clock data from: `{data_dir}/clock_output/human_rna_age_short.csv`
    * Create output directory if needed: `{results_dir}/{study_id}/`

2. Study Design Information:
   {study_design}

3. Clock Selection and Analysis:
    * **IMPORTANT: DO NOT compare young and old populations directly**
    * **Target Aging Clocks:**
    * `Hazard2_All_All_All_scaleddiff_EN_tAge`

    * **For Each Condition:**
    * Identify appropriate control group(s) based on metadata
    * Determine relevant covariates (age, sex, tissue type, batch effects)
    * Focus on comparing treatment vs. control within the same age group
    * Before statistical testing, determine the experiment's trust score (0-10):
        Trust score evaluates the experimental validity and translational potential of an aging study based on intervention/perturbation type and biological plausibility, completely independent of statistical outcomes like p-values and effect sizes.

        * **0-3: Low value**
        * Typically immortalized/cancer cell lines 
        * Studies with limited connection to aging biology
        * Minimal literature precedent for mechanism

        * **4-6: Moderate value**
        * Well-designed in vitro studies or simple model organisms
        * Reasonable connection to known aging pathways
        * Some supporting evidence in aging research

        * **7-8: High value**
        * In vivo studies or promising in vitro work with strong mechanistic basis
        * Clear connection to established aging biology
        * Substantial literature support
        * Good translational potential

        * **9-10: Exceptional value**
        * Novel interventions with compelling aging-related mechanisms
        * Robust evidence across experimental systems
        * Exceptional translational potential for human applications
    * Perform statistical analysis:
        * Use appropriate tests based on data structure (t-test, ANOVA, or linear models)
        * Report raw p-values (no adjustment required)
        * Calculate standardized effect sizes

4. Deliverables:
* Standardized results table with these columns:
    * gse_id: The dataset identifier ({study_id})
    * comparison_name: Descriptive name of the comparison (e.g., "Drug X vs Vehicle")
    * condition_name: Concise intervention name (e.g., "Rapamycin")
    * condition_category: Categorization (one of: Genetic, Drug, Diet, Environment, Disease, Other)
    * covariate: Variables controlled for in analysis
    * clock: Specific aging clock used
    * statistical_test: Method used (e.g., "ANOVA", "Linear Mixed Model")
    * p_value: Raw p-value (do NOT use adjusted p-values)
    * effect_size: Magnitude of effect (Cohen's d or similar)
    * direction: Age effect direction (1=accelerated aging, -1=decelerated aging)
    * trust_score: Evaluate experimental follow-up value (0-10):


Implementation Requirements:
    * Use Python with pandas, scipy, and statsmodels
    * Implement modular code structure with separate functions for:
    * Data loading and preprocessing
    * Statistical analysis
    * Results formatting
    * Include efficient documentation and inline comments
    * Handle missing data and outliers appropriately

Output Requirements
    * Save to: {results_dir}/{study_id}/aging_analysis.csv
    * DO NOT generate reports, visualizations, or any other output.
    * DO NOT provide explanations or summaries in the response.
    * Return ONLY a short confirmation message when complete.
'''

In [ ]:
out = agent.go(task_intention)

In [ ]:
pd.read_csv("../result_hs_rna/GSE99999/aging_analysis.csv")

In [ ]:
pd.read_csv("../result_hs_rna/GSE100007/aging_analysis.csv")